# KL-IG² Demo — KL-Descent Integrated Gradients

Rather than a fixed parametric path in (μ, σ) space, **KL-IG²** builds its
integration path adaptively by descending **KL(p(y|x) ‖ q)** in pixel space.
Attribution is the standard path integral of the KL gradient along this path:

$$\text{attr}_i = \sum_t \frac{\partial \text{KL}}{\partial x_i}(x_t) \cdot \Delta x_{t,i}$$

**Completeness:** $\sum_i \text{attr}_i \approx \text{KL}_0 - \text{KL}_T$

This notebook is a **100-sample pilot**: confirms the path descends KL,
checks completeness, and gives rough Insertion/Deletion + Sensitivity-n numbers
vs KL-IG (linear path) baseline.

In [ ]:
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1 /content/KLIG_V1 2>/dev/null || \
    git -C /content/KLIG_V1 fetch origin claude/general-session-FcgoB && \
    git -C /content/KLIG_V1 reset --hard origin/claude/general-session-FcgoB
!pip install -e /content/KLIG_V1/infocube-main -q
!pip install captum datasets tqdm scikit-learn -q

In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

for _root in ["/content/KLIG_V1/infocube-main", "infocube-main"]:
    if os.path.isdir(_root) and _root not in sys.path:
        sys.path.insert(0, _root)

from klig import KLIntegratedGradients, KLIG2Attributor
from torchvision.models import resnet50, ResNet50_Weights
print("imports OK")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
N_IMGS            = 100
N_STEPS_KLIG      = 50    # KL-IG integration steps
N_SAMPLES_KLIG    = 10    # KL-IG MC samples per step
T_KLIG2           = 50    # KL-IG² path steps
STEP_SIZE         = 0.01
KL_STOP           = 1e-3
SIGMA_FINAL       = 1 / 256
N_INSERTION_STEPS = 50
FORCE_RECOMPUTE   = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CACHE_DIR = Path("/content/drive/MyDrive/klig2_cache")
except Exception:
    CACHE_DIR = Path("klig2_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

METHODS = [
    "KL-IG (linear)",
    "KL-IG² (2nd class)",
    "KL-IG² (uniform)",
]
COLORS = {
    "KL-IG (linear)":     "#333333",
    "KL-IG² (2nd class)": "#e41a1c",
    "KL-IG² (uniform)":   "#377eb8",
}

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
weights    = ResNet50_Weights.IMAGENET1K_V2
model      = resnet50(weights=weights).to(DEVICE).eval()
preprocess = weights.transforms()
imagenet_labels = weights.meta["categories"]
print("ResNet50 loaded")

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def denormalize(x):
    return x.cpu() * _STD + _MEAN

def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0)
    return a.gather(0, idx.unsqueeze(0)).squeeze(0)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
_cache_ds = CACHE_DIR / "dataset.pkl"
if not FORCE_RECOMPUTE and _cache_ds.exists():
    with open(_cache_ds, "rb") as f: dataset = pickle.load(f)
    print(f"[cache] dataset n={len(dataset)}")
else:
    from datasets import load_dataset as _hf
    _ds = _hf("evanarlian/imagenet_1k_resized_256", split="train", streaming=True)
    dataset = []
    for item in tqdm(_ds.take(N_IMGS * 4), desc="loading"):
        img = item["image"]
        if img.mode != "RGB": img = img.convert("RGB")
        x = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(x)
            tgt  = int(logits.argmax(-1).item())
            conf = logits.softmax(-1)[0, tgt].item()
        if conf > 0.3:
            dataset.append({"x": x, "target": tgt, "idx": len(dataset)})
        if len(dataset) >= N_IMGS: break
    with open(_cache_ds, "wb") as f: pickle.dump(dataset, f)
    print(f"Collected {len(dataset)} images")

In [ ]:
# ── Attribution loop ──────────────────────────────────────────────────────────
_cache_attr = CACHE_DIR / "klig2_attrs.pkl"

if not FORCE_RECOMPUTE and _cache_attr.exists():
    with open(_cache_attr, "rb") as f:
        all_attrs, all_attrs_sq, all_path_meta = pickle.load(f)
    print("[cache] attrs loaded")
else:
    all_attrs     = {m: [] for m in METHODS}
    all_attrs_sq  = {m: [] for m in METHODS}   # KL-IG² squared form
    all_path_meta = []                          # kl_traj + path_len per image

    klig_baseline = KLIntegratedGradients(
        model, n_steps=N_STEPS_KLIG, n_samples=N_SAMPLES_KLIG,
        sigma_final=SIGMA_FINAL, device=DEVICE)

    klig2_2nd = KLIG2Attributor(
        model, T=T_KLIG2, step_size=STEP_SIZE, kl_stop=KL_STOP, device=DEVICE)
    klig2_uni = KLIG2Attributor(
        model, T=T_KLIG2, step_size=STEP_SIZE, kl_stop=KL_STOP, device=DEVICE)

    for row in tqdm(dataset, desc="attributing"):
        x, tgt = row["x"], row["target"]
        x1 = x.squeeze(0).to(DEVICE)
        meta = {}

        # KL-IG baseline
        r_klig = klig_baseline.attribute(x1, target=tgt)
        all_attrs["KL-IG (linear)"].append(absmax_collapse(r_klig.attr).cpu())
        all_attrs_sq["KL-IG (linear)"].append(None)

        # KL-IG² second_class
        r2 = klig2_2nd.attribute(x1, q_mode="second_class")
        all_attrs["KL-IG² (2nd class)"].append(absmax_collapse(r2.attr).cpu())
        all_attrs_sq["KL-IG² (2nd class)"].append(absmax_collapse(r2.attr_sq).cpu())
        meta["2nd_kl_traj"]  = r2.kl_traj
        meta["2nd_path_len"] = r2.path_len

        # KL-IG² uniform
        r3 = klig2_uni.attribute(x1, q_mode="uniform")
        all_attrs["KL-IG² (uniform)"].append(absmax_collapse(r3.attr).cpu())
        all_attrs_sq["KL-IG² (uniform)"].append(absmax_collapse(r3.attr_sq).cpu())
        meta["uni_kl_traj"]  = r3.kl_traj
        meta["uni_path_len"] = r3.path_len

        all_path_meta.append(meta)

    with open(_cache_attr, "wb") as f:
        pickle.dump((all_attrs, all_attrs_sq, all_path_meta), f)
    print("Done.")

In [ ]:
# ── 1. Path sanity: KL trajectories ──────────────────────────────────────────
N_SHOW = 8
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor="white")

for i in range(min(N_SHOW, len(all_path_meta))):
    traj_2nd = all_path_meta[i]["2nd_kl_traj"]
    traj_uni = all_path_meta[i]["uni_kl_traj"]
    axes[0].plot(traj_2nd, alpha=0.6, lw=1)
    axes[1].plot(traj_uni, alpha=0.6, lw=1)

for ax, title in zip(axes, ["KL-IG² (2nd class)", "KL-IG² (uniform)"]):
    ax.set_xlabel("Step"); ax.set_ylabel("KL")
    ax.set_title(f"{title} — KL trajectories (n={N_SHOW})")
    ax.axhline(KL_STOP, color="red", lw=1, ls="--", label=f"kl_stop={KL_STOP}")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Path sanity: KL should decrease monotonically", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

# path length histogram
lens_2nd = [m["2nd_path_len"] for m in all_path_meta]
lens_uni = [m["uni_path_len"] for m in all_path_meta]
fig, ax = plt.subplots(figsize=(7, 3), facecolor="white")
ax.hist(lens_2nd, bins=20, alpha=0.6, label="2nd class", color="#e41a1c")
ax.hist(lens_uni, bins=20, alpha=0.6, label="uniform",   color="#377eb8")
ax.axvline(T_KLIG2, color="black", lw=1, ls="--", label=f"T={T_KLIG2}")
ax.set_xlabel("Path length (steps)"); ax.set_ylabel("Count")
ax.set_title("Path length distribution")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"2nd class: mean={np.mean(lens_2nd):.1f}  max={max(lens_2nd)}")
print(f"uniform:   mean={np.mean(lens_uni):.1f}  max={max(lens_uni)}")

In [ ]:
# ── 2. Attribution maps (single image) ───────────────────────────────────────
VIS_IDX = 0
row0    = dataset[VIS_IDX]
img_np  = np.clip(denormalize(row0["x"][0]).permute(1,2,0).numpy(), 0, 1)

# signed attr + squared attr side by side for KL-IG²
show_maps = [
    ("KL-IG (linear)",     all_attrs["KL-IG (linear)"][VIS_IDX],        "RdBu_r"),
    ("KL-IG² 2nd (signed)", all_attrs["KL-IG² (2nd class)"][VIS_IDX],   "RdBu_r"),
    ("KL-IG² 2nd (sq)",    all_attrs_sq["KL-IG² (2nd class)"][VIS_IDX], "cividis"),
    ("KL-IG² uni (signed)", all_attrs["KL-IG² (uniform)"][VIS_IDX],     "RdBu_r"),
    ("KL-IG² uni (sq)",    all_attrs_sq["KL-IG² (uniform)"][VIS_IDX],   "cividis"),
]

fig, axes = plt.subplots(1, 1 + len(show_maps),
                          figsize=(3.0 * (1 + len(show_maps)), 3.2), facecolor="white")
axes[0].imshow(img_np); axes[0].axis("off")
axes[0].set_title("Original", fontsize=9, fontweight="bold")

for ax, (title, amap, cmap) in zip(axes[1:], show_maps):
    a    = amap.numpy()
    vmax = max(float(np.percentile(np.abs(a), 99)), 1e-9)
    ax.imshow(a, cmap=cmap, vmin=-vmax if cmap == "RdBu_r" else 0, vmax=vmax)
    ax.axis("off")
    ax.set_title(title, fontsize=8, fontweight="bold")

plt.suptitle(f"Attribution maps — {imagenet_labels[row0['target']]}",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ── 3. Completeness check ─────────────────────────────────────────────────────
# For KL-IG²: Σ attr ≈ KL_0 − KL_T
# For KL-IG:  Σ attr ≈ E[f(x_final)] − E[f(x_noise)]  (different completeness)
N_CC = 20
print(f"KL-IG² completeness (n={N_CC})")
print(f"  Σ_i attr_i  vs  KL_0 − KL_T\n")

errs_2nd, errs_uni = [], []
for i in range(min(N_CC, len(dataset))):
    meta = all_path_meta[i]

    kl0_2nd = meta["2nd_kl_traj"][0] if meta["2nd_kl_traj"] else 0.0
    klT_2nd = meta["2nd_kl_traj"][-1] if meta["2nd_kl_traj"] else 0.0
    expected_2nd = kl0_2nd - klT_2nd
    got_2nd = float(all_attrs["KL-IG² (2nd class)"][i].sum())
    errs_2nd.append(abs(got_2nd - expected_2nd) / (abs(expected_2nd) + 1e-8))

    kl0_uni = meta["uni_kl_traj"][0] if meta["uni_kl_traj"] else 0.0
    klT_uni = meta["uni_kl_traj"][-1] if meta["uni_kl_traj"] else 0.0
    expected_uni = kl0_uni - klT_uni
    got_uni = float(all_attrs["KL-IG² (uniform)"][i].sum())
    errs_uni.append(abs(got_uni - expected_uni) / (abs(expected_uni) + 1e-8))

print(f"  2nd class:  mean rel err = {np.mean(errs_2nd)*100:.2f}%")
print(f"  uniform:    mean rel err = {np.mean(errs_uni)*100:.2f}%")
print("  (target: <5%; >10% suggests discretization error)")

In [ ]:
# ── 4. Insertion / Deletion AUC ───────────────────────────────────────────────
_cache_id = CACHE_DIR / "klig2_ins_del.pkl"

def insertion_deletion(model, x, attr_map, target, n_steps=N_INSERTION_STEPS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    order = attr_map.detach().view(-1).abs().argsort(descending=True)
    pps   = max(1, H * W // n_steps)
    blur  = F.avg_pool2d(x, 31, 1, 15)
    x_ins, x_del = blur.clone(), x.clone()
    ins_s, del_s = [], []
    with torch.no_grad():
        for step in range(n_steps):
            pix = order[step * pps:(step + 1) * pps]
            for ch in range(C):
                x_ins[:, ch].reshape(-1)[pix] = x[:, ch].reshape(-1)[pix]
                x_del[:, ch].reshape(-1)[pix] = blur[:, ch].reshape(-1)[pix]
            ins_s.append(model(x_ins).softmax(-1)[0, target].item())
            del_s.append(model(x_del).softmax(-1)[0, target].item())
    return float(np.trapz(ins_s) / n_steps), float(np.trapz(del_s) / n_steps)

if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id, "rb") as f: ins_auc, del_auc = pickle.load(f)
else:
    ins_auc = defaultdict(list); del_auc = defaultdict(list)
    for row in tqdm(dataset[:N_IMGS], desc="ins/del"):
        x, tgt = row["x"], row["target"]
        for m in METHODS:
            attr = all_attrs[m][row["idx"]].to(DEVICE).unsqueeze(0)
            i, d = insertion_deletion(model, x, attr, tgt)
            ins_auc[m].append(i); del_auc[m].append(d)
    ins_auc, del_auc = dict(ins_auc), dict(del_auc)
    with open(_cache_id, "wb") as f: pickle.dump((ins_auc, del_auc), f)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor="white")
for ax, (title, aucs) in zip(axes, [("Insertion AUC ↑", ins_auc), ("Deletion AUC ↓", del_auc)]):
    for xi, m in enumerate(METHODS):
        v = aucs[m]; mu = np.mean(v); ci = 1.96 * np.std(v) / len(v) ** 0.5
        ax.bar(xi, mu, color=COLORS[m], alpha=0.85, width=0.6)
        ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
    ax.set_xticks(range(len(METHODS)))
    ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=9)
    ax.set_title(title)
plt.suptitle(f"Insertion / Deletion AUC  (n={N_IMGS})", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Sensitivity-n (signed) ─────────────────────────────────────────────────
_cache_sn = CACHE_DIR / "klig2_sens_n.pkl"

def sensitivity_n(model, x, attr_map, target, n_subsets=50, subset_size=0.1):
    rng = np.random.default_rng(42)
    attr_flat = attr_map.cpu().detach().view(-1).numpy()
    n_pix = attr_flat.size
    n_sel = max(1, int(n_pix * subset_size))
    corrs = []
    with torch.no_grad():
        f_x = model(x).softmax(-1)[0, target].item()
        for _ in range(n_subsets):
            idx    = rng.choice(n_pix, n_sel, replace=False)
            x_mask = x.clone()
            for ch in range(x.shape[1]):
                x_mask[:, ch].reshape(-1)[idx] = 0
            f_mask  = model(x_mask).softmax(-1)[0, target].item()
            delta_f = f_x - f_mask
            delta_a = float(attr_flat[idx].sum())   # signed
            corrs.append((delta_f, delta_a))
    df = np.array([c[0] for c in corrs])
    da = np.array([c[1] for c in corrs])
    if df.std() < 1e-9 or da.std() < 1e-9: return 0.0
    return float(np.corrcoef(df, da)[0, 1])

if not FORCE_RECOMPUTE and _cache_sn.exists():
    with open(_cache_sn, "rb") as f: sens_n = pickle.load(f)
else:
    sens_n = defaultdict(list)
    for row in tqdm(dataset, desc="sensitivity-n"):
        x, tgt = row["x"], row["target"]
        for m in METHODS:
            attr = all_attrs[m][row["idx"]].to(DEVICE).unsqueeze(0)
            sens_n[m].append(sensitivity_n(model, x, attr, tgt))
    sens_n = dict(sens_n)
    with open(_cache_sn, "wb") as f: pickle.dump(sens_n, f)

fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
for xi, m in enumerate(METHODS):
    v = sens_n[m]; mu = np.mean(v); ci = 1.96 * np.std(v) / len(v) ** 0.5
    ax.bar(xi, mu, color=COLORS[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Pearson r (signed)"); ax.set_title(f"Sensitivity-n  (n={len(dataset)})")
plt.tight_layout(); plt.show()

In [ ]:
# ── 6. Sparsity (Gini) ────────────────────────────────────────────────────────
def gini(v):
    v = v.abs().flatten().numpy()
    v = np.sort(v); n = len(v)
    return float((2 * np.arange(1, n + 1) - n - 1) @ v / (n * v.sum() + 1e-12))

gini_scores = {m: [gini(all_attrs[m][i]) for i in range(len(dataset))] for m in METHODS}

fig, ax = plt.subplots(figsize=(7, 4), facecolor="white")
for xi, m in enumerate(METHODS):
    v = gini_scores[m]; mu = np.mean(v); ci = 1.96 * np.std(v) / len(v) ** 0.5
    ax.bar(xi, mu, color=COLORS[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
ax.set_xticks(range(len(METHODS)))
ax.set_xticklabels(METHODS, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Gini coefficient ↑"); ax.set_title(f"Sparsity  (n={len(dataset)})")
plt.tight_layout(); plt.show()

In [ ]:
# ── 7. Summary table ──────────────────────────────────────────────────────────
import pandas as pd

ci95 = lambda v: 1.96 * np.std(v) / len(v) ** 0.5
rows_sum = []
for m in METHODS:
    row_s = {"Method": m}
    if gini_scores.get(m):
        v = gini_scores[m]; row_s["Gini ↑"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if ins_auc.get(m):
        v = ins_auc[m];     row_s["Ins AUC ↑"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if del_auc.get(m):
        v = del_auc[m];     row_s["Del AUC ↓"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    if sens_n.get(m):
        v = sens_n[m];      row_s["Sens-n ↑"] = f"{np.mean(v):.3f}±{ci95(v):.3f}"
    rows_sum.append(row_s)

df_sum = pd.DataFrame(rows_sum).set_index("Method")
print(df_sum.to_string())
df_sum